# Methodology OverviewComprehensive documentation of the ODE-Multiomics Benchmark experimental design.

In [ ]:
import syssys.path.insert(0, '..')import numpy as npimport pandas as pdfrom src.reynolds_ode import REYNOLDS_PARAMS

## Reynolds ODE Model ParametersThe benchmark is based on the Reynolds et al. (2006) acute inflammation model.

In [ ]:
# Display parameters as a structured tableparam_categories = {    'Pathogen Dynamics': ['k_pg', 'P_inf', 'k_pm', 's_dm', 'mu_p'],    'Pro-inflammatory (N*)': ['s_nr', 'epsilon_nr', 's_nm', 's_nd', 'N_inf', 'mu_nr'],    'DAMP (D)': ['k_dn', 'k_df', 'mu_d'],    'Anti-inflammatory (CA)': ['s_c', 's_cn', 'mu_c'],    'Tissue Damage': ['k_f', 'k_fh']}for category, params in param_categories.items():    print(f'\n{category}:')    print('-' * 50)    for param in params:        value = REYNOLDS_PARAMS[param]        print(f'  {param:15s} = {value:10.4f}')

## Synthetic Data GenerationHow synthetic patients are created with controlled heterogeneity.

In [ ]:
print('Synthetic Data Generation Strategy:')print('=' * 60)print()print('1. PARAMETER NOISE')print('   - Lognormal distribution: sigma = 0.15 (plus/minus 15% log-scale)')print('   - Per-patient parameters: p_i = p_base * exp(N(0, 0.15))')print('   - Reflects natural inter-patient variability')print()print('2. MEASUREMENT NOISE')print('   - Gaussian proportional: sigma = 0.10 (10% coefficient of variation)')print('   - Observation model: y_obs = y_true * (1 + N(0, 0.10))')print('   - Realistic sensor/assay noise')print()print('3. OBSERVATION TIMEPOINTS')timepoints = np.array([0, 6, 12, 24, 48, 72])print(f'   - Timepoints (hours): {timepoints}')print(f'   - Number of measurements: {len(timepoints)}')print(f'   - Clinical schedule: 0, 6h, 12h, 1d, 2d, 3d')print()print('4. OUTCOME STRATIFICATION')print('   - Resolution:  low P0  (0.5 - 3.5)  -> antibodies clear pathogen')print('   - Chronic:     mid P0  (4.0 - 9.0)  -> persistent inflammation')print('   - Death:      high P0 (11.0-20.0)  -> runaway inflammation')

## MOTIF PipelineODE-based Multi-Omics Temporal Integration Framework

In [ ]:
print('MOTIF Pipeline Overview:')print('=' * 60)print()print('STEP 1: ODE CALIBRATION')print('  - Optimize parameters: k_pg, mu_p, s_nr, s_c, P0')print('  - Method: L-BFGS-B optimization')print('  - Objective: Minimize squared residuals on observed (N*, CA, f)')print('  - Restarts: 5+1 to escape local minima')print()print('STEP 2: PROXY GENERATION')print('  - Simulate calibrated ODE with fitted parameters')print('  - Extract latent trajectories: P(t), D(t), h(t)')print('  - Compute summary statistics: AUC, peak, final values')print()print('STEP 3: FEATURE EXTRACTION')print('  - Observed features: AUC, peak, final for N*, CA, f')print('  - Proxy features: AUC, peak, final for P, D, h')print('  - Total features: ~18 dimensions')print()print('STEP 4: CORRELATION ANALYSIS')print('  - Spearman rank correlation: features vs outcome')print('  - Identify strongest biomarkers')print()print('STEP 5: CLASSIFICATION')print('  - Logistic regression: features -> outcome')print('  - Evaluate: AUROC, F1 score')print('  - Compare: observed features vs with proxies')

## UDE PipelineUniversal Differential Equations + Symbolic Regression

In [ ]:
print('UDE Pipeline Overview:')print('=' * 60)print()print('STEP 1: NEURAL NETWORK ARCHITECTURE')print('  - Input: Observed state [N*, CA, f]')print('  - Hidden layers: 2 x tanh')print('  - Output: Scalar clearance rate correction')print('  - Non-linearity: Softplus (ensures non-negative rates)')print()print('STEP 2: HYBRID ODE SYSTEM')print('  - Known dynamics: 4 of 5 equations from Reynolds model')print('  - Learned dynamics: Pathogen clearance theta(N*, CA, f)')print('  - State: [P, N*, D, CA, f]')print('  - Training integrator: torchdiffeq with RK4')print()print('STEP 3: TRAINING')print('  - Loss: MSE on observed variables')print('  - Optimizer: Adam')print('  - LR schedule: ReduceLROnPlateau')print('  - Typical: 50-500 epochs, batch size 4-16')print()print('STEP 4: SYMBOLIC REGRESSION (OPTIONAL)')print('  - Method: SINDy (Sparse Identification of Nonlinear Dynamics)')print('  - Extract: Interpretable equations from learned function')print('  - e.g., theta(N*) = c0 + c1*N* + c2*N*^2 + ...')

## Experimental ConditionsSix benchmark experiments testing different robustness scenarios.

In [ ]:
print('Benchmark Experiments:')print('=' * 70)print()experiments = [    {        'name': 'Baseline',        'description': 'Standard setup with nominal noise levels',        'param_noise': '15%',        'obs_noise': '10%',        'timepoints': 6    },    {        'name': 'Sparse Data',        'description': 'Reduced measurement timepoints',        'param_noise': '15%',        'obs_noise': '10%',        'timepoints': 3    },    {        'name': 'High Noise',        'description': 'Increased observation noise',        'param_noise': '15%',        'obs_noise': '25%',        'timepoints': 6    },    {        'name': 'Misspecified ODE',        'description': 'Model mismatch in known dynamics',        'param_noise': '15%',        'obs_noise': '10%',        'timepoints': 6    },    {        'name': 'Reduced Sampling',        'description': 'Limited patient cohort',        'param_noise': '15%',        'obs_noise': '10%',        'timepoints': 6    },    {        'name': 'Multi-resolution',        'description': 'Heterogeneous timepoint schedules',        'param_noise': '15%',        'obs_noise': '10%',        'timepoints': 'variable'    }]for i, exp in enumerate(experiments, 1):    print(f'{i}. {exp["name"]}')    print(f'   {exp["description"]}')    print(f'   Param noise: {exp["param_noise"]}, Obs noise: {exp["obs_noise"]}, Timepoints: {exp["timepoints"]}')    print()

## Evaluation MetricsHow methods are compared and evaluated.

In [ ]:
print('Evaluation Metrics:')print('=' * 70)print()print('1. STATE VARIABLE RECOVERY')print('   Metric: R-squared = 1 - (SS_residual / SS_total)')print('   Computed for: P (pathogen), D (DAMP), h (tissue damage)')print('   Scale: 0-1 (1 = perfect recovery)')print()print('2. OUTCOME CLASSIFICATION')print('   Metric: AUROC (Area Under ROC Curve)')print('   Task: Predict outcome (resolution/chronic/death)')print('   One-vs-rest approach for multiclass')print()print('3. FEATURE IMPORTANCE')print('   Metric: Correlation (Spearman) with outcome')print('   Identifies biomarkers')print()print('4. COMPUTATIONAL EFFICIENCY')print('   Metric: Wall-clock time per patient')print('   MOTIF: ~1-5s (calibration + proxies)')print('   UDE: ~100-500s (depends on epochs)')print()print('5. INTERPRETABILITY')print('   MOTIF: Fitted parameter values are interpretable')print('   UDE: Learned function requires SINDy for interpretation')

## ReproducibilityGuide to reproducing the benchmark results.

In [ ]:
print('Reproducibility Guide:')print('=' * 70)print()print('1. SEEDING STRATEGY')print('   - Master seed: Set before cohort generation')print('   - Patient seed: master_seed + patient_id')print('   - Ensures: Same P0 and parameters for same patient across runs')print()print('2. RUN EXPERIMENTS')print('   Command line:')print('   $ python src/run_experiment.py --config config_baseline.yaml')print()print('   Arguments:')print('   --config:  YAML file with experiment parameters')print('   --seed:    Random seed (default 42)')print('   --output:  Results directory (default results/)')print()print('3. EXPECTED OUTPUTS')print('   - results/{timestamp}/synthetic_cohort.pkl')print('   - results/{timestamp}/motif_calibrations.pkl')print('   - results/{timestamp}/ude_model.pt')print('   - results/{timestamp}/metrics.json')print()print('4. HARDWARE CONSIDERATIONS')print('   - GPU: Recommended for UDE training (much faster)')print('   - CPU: ~30-60min for full benchmark')print('   - RAM: ~2GB for 500 patients')print()print('5. DEPENDENCIES')print('   - Core: numpy, scipy, scikit-learn, matplotlib')print('   - ODE: torch, torchdiffeq')print('   - Optional: pysindy (symbolic regression)')

## Mathematical DetailsKey equations and derivations.

In [ ]:
print('Reynolds ODE System:')print('=' * 70)print()print('State variables:')print('  P:     Pathogen load')print('  N*:    Early pro-inflammatory mediator')print('  D:     Late pro-inflammatory/DAMP mediator')print('  CA:    Anti-inflammatory mediator (antibodies)')print('  f:     Tissue damage fraction')print('  h:     Tissue health = 1 - f')print()print('Core dynamics:')print('  dP/dt   = k_pg*P*(1 - P/P_inf) - k_pm*N*/(s_dm + N*)*P - mu_p*P')print('  dN*/dt  = s_nr/(1 + (CA/eps_nr)^2)*[P/(s_nm+P) + D/(s_nd+D)]*(1-N*/N_inf) - mu_nr*N*')print('  dD/dt   = k_dn*N* + k_df*f - mu_d*D')print('  dCA/dt  = s_c*N*^2/(s_cn^2 + N*^2) - mu_c*CA')print('  df/dt   = k_f*N**(1-f) - k_fh*h*f')print()print('Key features:')print('  - Negative feedback: CA suppresses N* production (epsilon_nr term)')print('  - Pathogen-driven damage: N* drives both D and f')print('  - Recovery mechanism: Tissue repair (k_fh) and pathogen clearance')

## References- Reynolds et al. (2006). A reduced mathematical model of acute inflammatory response. J Theor Biol.- Funk et al. (2025). MOTIF: Multi-Omics Temporal Integration Framework.- Rackauckas & Nie (2020). Universal Differential Equations. arXiv.- Brunton et al. (2016). Discovering governing equations from data. PNAS.For more information, see README.md and TECHNICAL_SPEC.md